# Streamlit Preparation for CBIR

This notebook prepares all required data files for Streamlit deployment.

It will:
- load the processed dataset
- extract features for all 4 algorithms
- save feature databases
- save image paths and labels

These saved files will later be loaded directly in the Streamlit app.

In [1]:
import os
import cv2
import pickle
import numpy as np
from skimage.feature import graycomatrix, graycoprops

In [2]:
DATASET_PATH = "processed_dataset"
SAVE_PATH = "saved_features"

os.makedirs(SAVE_PATH, exist_ok=True)

categories = sorted(os.listdir(DATASET_PATH))
print("Categories:", categories)

Categories: ['ambulance', 'bicycle', 'bus', 'car', 'fire_truck', 'motorcycle', 'tractor', 'truck', 'van']


## Collect Image Paths and Labels

In [3]:
all_image_paths = []
all_image_labels = []

for category in categories:
    category_path = os.path.join(DATASET_PATH, category)

    for img_name in os.listdir(category_path):
        img_path = os.path.join(category_path, img_name)
        all_image_paths.append(img_path)
        all_image_labels.append(category)

print("Total images:", len(all_image_paths))

Total images: 1800


## Define Feature Extraction Functions

### Color Histogram

In [4]:
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)

    if image is None:
        return None

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    hist = cv2.calcHist(
        [image], [0, 1, 2], None, bins,
        [0, 256, 0, 256, 0, 256]
    )

    hist = cv2.normalize(hist, hist).flatten()
    return hist

### GLCM

In [5]:
def extract_glcm_features(image_path, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    glcm = graycomatrix(
        image,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(glcm, 'contrast').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()

    return np.hstack([contrast, correlation, energy, homogeneity])

### Hu Moments

In [6]:
def extract_hu_moments(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    blurred = cv2.GaussianBlur(image, (5, 5), 0)

    _, thresh = cv2.threshold(
        blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    moments = cv2.moments(thresh)
    hu = cv2.HuMoments(moments).flatten()

    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
    return hu

### ORB

In [7]:
orb = cv2.ORB_create(nfeatures=1000)

def extract_orb_descriptors(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    keypoints, descriptors = orb.detectAndCompute(image, None)
    return descriptors

## Build and Save Color Histogram Features

In [8]:
color_features = []

for img_path in all_image_paths:
    feature = extract_color_histogram(img_path)
    color_features.append(feature)

color_features = np.array(color_features)

with open(os.path.join(SAVE_PATH, "color_features.pkl"), "wb") as f:
    pickle.dump(color_features, f)

print("Color features saved:", color_features.shape)

Color features saved: (1800, 512)


## Build and Save GLCM Features

In [9]:
glcm_features = []

for img_path in all_image_paths:
    feature = extract_glcm_features(img_path)
    glcm_features.append(feature)

glcm_features = np.array(glcm_features)

with open(os.path.join(SAVE_PATH, "glcm_features.pkl"), "wb") as f:
    pickle.dump(glcm_features, f)

print("GLCM features saved:", glcm_features.shape)

GLCM features saved: (1800, 16)


## Build and Save Hu Moments Features

In [10]:
hu_features = []

for img_path in all_image_paths:
    feature = extract_hu_moments(img_path)
    hu_features.append(feature)

hu_features = np.array(hu_features)

with open(os.path.join(SAVE_PATH, "hu_features.pkl"), "wb") as f:
    pickle.dump(hu_features, f)

print("Hu features saved:", hu_features.shape)

Hu features saved: (1800, 7)


## Build and Save ORB Descriptors

In [11]:
orb_descriptors_db = []

for img_path in all_image_paths:
    desc = extract_orb_descriptors(img_path)
    orb_descriptors_db.append(desc)

with open(os.path.join(SAVE_PATH, "orb_descriptors.pkl"), "wb") as f:
    pickle.dump(orb_descriptors_db, f)

print("ORB descriptors saved.")

ORB descriptors saved.


## Save Image Paths and Labels

In [12]:
with open(os.path.join(SAVE_PATH, "image_paths.pkl"), "wb") as f:
    pickle.dump(all_image_paths, f)

with open(os.path.join(SAVE_PATH, "image_labels.pkl"), "wb") as f:
    pickle.dump(all_image_labels, f)

print("Image paths and labels saved.")

Image paths and labels saved.


## Verify Saved Files

In [13]:
print("Saved files:")
for file_name in os.listdir(SAVE_PATH):
    print("-", file_name)

Saved files:
- color_features.pkl
- glcm_features.pkl
- hu_features.pkl
- image_labels.pkl
- image_paths.pkl
- orb_descriptors.pkl


In [14]:
import pickle

with open("saved_features/color_features.pkl", "rb") as f:
    color = pickle.load(f)

print(type(color), color.shape)

<class 'numpy.ndarray'> (1800, 512)


## Summary

This notebook prepares all required files for Streamlit deployment.

The following files are saved:
- color_features.pkl
- glcm_features.pkl
- hu_features.pkl
- orb_descriptors.pkl
- image_paths.pkl
- image_labels.pkl

These files will be loaded directly by the Streamlit app
to make retrieval faster and more efficient.